# 15. Samplers and solvers — DPM-Solver++ and complete UniPC

The old analytic `toy_data_prediction` oracle is removed. A small trainable data-prediction network is first optimized for five CPU steps on a VP diffusion path. DPM-Solver++ and UniPC then use that learned model.

UniPC keeps both halves of the reference algorithm: **UniP predictor + fresh endpoint model evaluation + UniC corrector**. Only data dimension, hidden width, batch size, and history length are reduced.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(5)
device = torch.device("cpu")
print("device:", device)


## 1. VP diffusion schedule


In [ ]:
def alpha(t):
    return torch.cos(0.5 * math.pi * t)


def sigma(t):
    return torch.sin(0.5 * math.pi * t)


def lambda_t(t):
    return torch.log(alpha(t)) - torch.log(sigma(t))


def inverse_lambda(value):
    return 2.0 / math.pi * torch.atan(torch.exp(-value))


## 2. Learned data-prediction model and five-step CPU check

DPM-Solver++ and the `predict_x0=True` UniPC form both consume data predictions. This model predicts `x0` from `(x_t,t)` instead of using a closed-form placeholder.


In [ ]:
class TinyDataPredictor(nn.Module):
    def __init__(self, data_dim=2, hidden_dim=24):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(data_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, x_t, t):
        model_input = torch.cat([x_t, t[:, None]], dim=-1)
        return self.net(model_input)


model = TinyDataPredictor().to(device)
clean = torch.randn(16, 2, device=device)
noise = torch.randn_like(clean)
train_t = torch.linspace(0.05, 0.95, 16, device=device)
noisy = (
    alpha(train_t)[:, None] * clean
    + sigma(train_t)[:, None] * noise
)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_history = []

for step in range(5):
    optimizer.zero_grad()
    prediction = model(noisy, train_t)
    loss = F.mse_loss(prediction, clean)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    print(f"step {step + 1}: data-prediction loss={loss.item():.6f}")

print("loss history:", loss_history)


def model_data_prediction(x, t):
    if t.ndim == 0:
        t = t.expand(x.size(0))
    return model(x, t)


## 3. DPM-Solver++ first- and second-order updates


In [ ]:
def dpmpp_first_order(x_s, s, t, model_fn):
    h = lambda_t(t) - lambda_t(s)
    model_s = model_fn(x_s, s)
    phi_1 = torch.expm1(-h)
    return (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * phi_1 * model_s
    )


def dpmpp_second_order(x_s, s, t, model_fn, r1=0.5):
    lambda_s = lambda_t(s)
    h = lambda_t(t) - lambda_s
    s1 = inverse_lambda(lambda_s + r1 * h)

    model_s = model_fn(x_s, s)
    x_s1 = (
        sigma(s1) / sigma(s) * x_s
        - alpha(s1) * torch.expm1(-r1 * h) * model_s
    )
    model_s1 = model_fn(x_s1, s1)

    phi_1 = torch.expm1(-h)
    correction = (
        0.5
        / r1
        * alpha(t)
        * phi_1
        * (model_s1 - model_s)
    )
    return (
        sigma(t) / sigma(s) * x_s
        - alpha(t) * phi_1 * model_s
        - correction
    )


x_s = torch.randn(2, 2, device=device)
s = torch.tensor(0.80, device=device)
t = torch.tensor(0.60, device=device)
print("DPM++ first order:", dpmpp_first_order(x_s, s, t, model_data_prediction))
print("DPM++ second order:", dpmpp_second_order(x_s, s, t, model_data_prediction))


## 4. Shared UniPC B(h) coefficient construction

The reference implementation works in log-SNR coordinates. Previous model outputs are converted into normalized first differences `D1`; the `R` matrix and phi recursion determine predictor/corrector coefficients.


In [ ]:
def unipc_bh_system(
    current_time,
    target_time,
    model_history,
    time_history,
    order,
    sample,
):
    model_s0 = model_history[-1]
    lambda_s0 = lambda_t(current_time)
    lambda_target = lambda_t(target_time)
    h = lambda_target - lambda_s0

    rks = []
    d1_terms = []
    for history_offset in range(1, order):
        previous_time = time_history[-(history_offset + 1)]
        previous_model = model_history[-(history_offset + 1)]
        lambda_previous = lambda_t(previous_time)

        rk = (lambda_previous - lambda_s0) / h
        rks.append(rk)
        d1_terms.append((previous_model - model_s0) / rk)

    rks.append(torch.ones((), device=sample.device))
    rks = torch.stack(rks)

    hh = -h
    h_phi_1 = torch.expm1(hh)
    h_phi_k = h_phi_1 / hh - 1.0
    b_h = torch.expm1(hh)

    factorial = 1.0
    matrix_rows = []
    rhs = []
    for power in range(1, order + 1):
        matrix_rows.append(rks.pow(power - 1))
        rhs.append(h_phi_k * factorial / b_h)

        factorial *= power + 1
        h_phi_k = h_phi_k / hh - 1.0 / factorial

    return {
        "model_s0": model_s0,
        "h_phi_1": h_phi_1,
        "b_h": b_h,
        "matrix": torch.stack(matrix_rows),
        "rhs": torch.stack(rhs),
        "d1_terms": d1_terms,
    }


## 5. UniP predictor

For order 2 the reference implementation uses the simplified predictor coefficient `rho=1/2`. Other multistep orders solve the reduced coefficient system.


In [ ]:
def unip_bh_predict(
    sample,
    current_time,
    target_time,
    model_history,
    time_history,
    order,
):
    system = unipc_bh_system(
        current_time,
        target_time,
        model_history,
        time_history,
        order,
        sample,
    )

    d1_terms = system["d1_terms"]
    if d1_terms:
        differences = torch.stack(d1_terms, dim=0)

        if order == 2:
            rho = torch.full(
                (1,), 0.5, dtype=sample.dtype, device=sample.device
            )
        else:
            rho = torch.linalg.solve(
                system["matrix"][:-1, :-1],
                system["rhs"][:-1],
            ).to(sample.dtype)

        predictor_residual = torch.einsum(
            "k,kbd->bd", rho, differences
        )
    else:
        predictor_residual = torch.zeros_like(sample)

    base = (
        sigma(target_time) / sigma(current_time) * sample
        - alpha(target_time)
        * system["h_phi_1"]
        * system["model_s0"]
    )
    return (
        base
        - alpha(target_time)
        * system["b_h"]
        * predictor_residual
    )


## 6. UniC corrector

UniC evaluates the diffusion model at the newly predicted endpoint. Its coefficient vector uses the full `R` system and includes the fresh difference `model_t - model_s0`. This is the part that distinguishes a complete UniPC step from UniP alone.


In [ ]:
def unic_bh_correct(
    last_sample,
    predicted_sample,
    current_time,
    target_time,
    model_history,
    time_history,
    target_model_output,
    order,
):
    system = unipc_bh_system(
        current_time,
        target_time,
        model_history,
        time_history,
        order,
        last_sample,
    )

    if order == 1:
        rho = torch.full(
            (1,), 0.5, dtype=last_sample.dtype, device=last_sample.device
        )
    else:
        rho = torch.linalg.solve(
            system["matrix"],
            system["rhs"],
        ).to(last_sample.dtype)

    d1_terms = system["d1_terms"]
    if d1_terms:
        previous_residual = torch.einsum(
            "k,kbd->bd",
            rho[:-1],
            torch.stack(d1_terms, dim=0),
        )
    else:
        previous_residual = torch.zeros_like(last_sample)

    endpoint_difference = (
        target_model_output - system["model_s0"]
    )
    correction_residual = (
        previous_residual + rho[-1] * endpoint_difference
    )

    base = (
        sigma(target_time) / sigma(current_time) * last_sample
        - alpha(target_time)
        * system["h_phi_1"]
        * system["model_s0"]
    )
    return (
        base
        - alpha(target_time)
        * system["b_h"]
        * correction_residual
    )


## 7. Complete UniPC multistep execution

Every step performs `UniP -> model(target) -> UniC`. The order grows with available history up to 3.


In [ ]:
sample = torch.randn(2, 2, device=device)
times = [
    torch.tensor(value, device=device)
    for value in [0.90, 0.75, 0.60, 0.45]
]
model_history = []
time_history = []

for step_index in range(len(times) - 1):
    current_time = times[step_index]
    target_time = times[step_index + 1]

    current_model = model_data_prediction(sample, current_time).detach()
    model_history.append(current_model)
    time_history.append(current_time)

    order = min(3, len(model_history))
    predicted_sample = unip_bh_predict(
        sample,
        current_time,
        target_time,
        model_history,
        time_history,
        order,
    )
    target_model = model_data_prediction(
        predicted_sample,
        target_time,
    ).detach()
    sample = unic_bh_correct(
        sample,
        predicted_sample,
        current_time,
        target_time,
        model_history,
        time_history,
        target_model,
        order,
    )

    print(
        f"UniPC step {step_index + 1}: order={order}, "
        f"sample_norm={sample.norm().item():.6f}"
    )


## References and provenance

- DPM-Solver++: data-prediction formulation and analytical log-SNR updates.
- UniPC reference implementation / Diffusers `UniPCMultistepScheduler`: B(h) predictor and corrector, normalized log-SNR history, first differences, phi recursion, coefficient linear systems, and fresh endpoint model evaluation.

Only tensor scale and training budget are reduced; the previous toy model oracle and predictor-only shortcut are removed.
